In [40]:
"""
Black-Scholes Live Option Surface Builder
Fetches real options data from yfinance and computes implied volatility surfaces.
Output format matches the synthetic all_bs_surfaces.csv exactly.
"""

import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
from datetime import datetime, date
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

try:
    import yfinance as yf
except ImportError:
    raise ImportError("yfinance is required. Install with: pip install yfinance")


class BlackScholesLiveSurface:
    """
    Build implied volatility surfaces from real options data via yfinance.
    Output matches the format of all_bs_surfaces.csv produced by the synthetic generator.
    """

    def __init__(self,
                 tickers: list = None,
                 risk_free_rate: float = 0.05,
                 output_dir: str = '../../final_dataset/live_option/black_scholes',
                 min_volume: int = 0,
                 min_open_interest: int = 0):
        """
        Parameters
        ----------
        tickers           : list of Yahoo Finance ticker symbols
        risk_free_rate    : annualised risk-free rate (e.g. 0.05 = 5 %)
        output_dir        : directory for CSV / pickle outputs
        min_volume        : minimum daily volume filter on option contracts
        min_open_interest : minimum open interest filter on option contracts
        """
        self.tickers = tickers or ['NFLX', 'SPOT', 'DIS']
        self.risk_free_rate = risk_free_rate
        self.output_dir = output_dir
        self.min_volume = min_volume
        self.min_open_interest = min_open_interest
        self.valuation_date = date.today().isoformat()

        os.makedirs(output_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # Black-Scholes helpers
    # ------------------------------------------------------------------

    def _bs_call(self, S, K, T, r, sigma):
        """Black-Scholes call price."""
        if T <= 0 or sigma <= 0:
            return max(0.0, S - K)
        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def _bs_put(self, S, K, T, r, sigma):
        """Black-Scholes put price via put-call parity."""
        return self._bs_call(S, K, T, r, sigma) - S + K * np.exp(-r * T)

    def _implied_vol(self, market_price, S, K, T, r, option_type='call'):
        """Newton-Brent implied volatility solver."""
        if T <= 0 or market_price <= 0:
            return np.nan

        intrinsic = max(0.0, S - K) if option_type == 'call' else max(0.0, K - S)
        if market_price < intrinsic - 1e-4:
            return np.nan

        def objective(sigma):
            if option_type == 'call':
                return self._bs_call(S, K, T, r, sigma) - market_price
            return self._bs_put(S, K, T, r, sigma) - market_price

        try:
            return brentq(objective, 1e-4, 5.0, xtol=1e-6, maxiter=200)
        except Exception:
            return np.nan

    def _greeks(self, S, K, T, r, sigma):
        """Return (delta_call, gamma, vega, theta_call)."""
        if T <= 0 or sigma <= 0:
            return np.nan, np.nan, np.nan, np.nan
        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        delta_call = norm.cdf(d1)
        gamma      = norm.pdf(d1) / (S * sigma * np.sqrt(T))
        vega       = S * norm.pdf(d1) * np.sqrt(T)
        theta_call = (
            -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        )
        return delta_call, gamma, vega, theta_call

    # ------------------------------------------------------------------
    # yfinance data fetch
    # ------------------------------------------------------------------

    def _fetch_spot(self, ticker: str) -> float:
        tk = yf.Ticker(ticker)
        hist = tk.history(period='1d')
        if hist.empty:
            raise ValueError(f"Could not fetch spot price for {ticker}")
        return float(hist['Close'].iloc[-1])

    def _fetch_expiries(self, ticker: str) -> tuple:
        """Return (yf.Ticker, list_of_expiry_strings)."""
        tk = yf.Ticker(ticker)
        return tk, list(tk.options)

    def _expiry_to_years(self, expiry_str: str) -> tuple:
        """Convert 'YYYY-MM-DD' string to (maturity_years, maturity_days)."""
        exp = datetime.strptime(expiry_str, '%Y-%m-%d').date()
        today = date.today()
        days = (exp - today).days
        years = days / 365.0
        return years, days

    # ------------------------------------------------------------------
    # Core surface builder for one ticker
    # ------------------------------------------------------------------

    def _build_surface(self, ticker: str, common_expiries: list) -> pd.DataFrame:
        """
        Fetch option chains for `common_expiries` and compute the IV surface.
        Returns a DataFrame in the same column order as all_bs_surfaces.csv.
        """
        print(f"\n  Processing {ticker} ...")
        tk = yf.Ticker(ticker)
        S0 = self._fetch_spot(ticker)
        print(f"    Spot price : {S0:.2f}")

        rows = []
        r = self.risk_free_rate

        for expiry in common_expiries:
            T_years, T_days = self._expiry_to_years(expiry)
            if T_years <= 0:
                continue

            try:
                chain = tk.option_chain(expiry)
            except Exception as e:
                print(f"    WARNING: could not fetch chain for {expiry}: {e}")
                continue

            calls = chain.calls.copy()
            puts  = chain.puts.copy()

            # Apply liquidity filters
            if self.min_volume > 0:
                calls = calls[calls['volume'].fillna(0) >= self.min_volume]
                puts  = puts[puts['volume'].fillna(0) >= self.min_volume]
            if self.min_open_interest > 0:
                calls = calls[calls['openInterest'].fillna(0) >= self.min_open_interest]
                puts  = puts[puts['openInterest'].fillna(0) >= self.min_open_interest]

            # Use mid-price; fall back to lastPrice if bid/ask unavailable
            def mid(df):
                bid = df['bid'].fillna(0)
                ask = df['ask'].fillna(0)
                mid_price = np.where((bid > 0) & (ask > 0), (bid + ask) / 2, df['lastPrice'])
                return mid_price

            calls = calls.copy()
            calls['mid'] = mid(calls)
            puts  = puts.copy()
            puts['mid']  = mid(puts)

            # Build a strike universe from both legs
            call_strikes = set(calls['strike'].values)
            put_strikes  = set(puts['strike'].values)
            all_strikes  = sorted(call_strikes | put_strikes)

            for K in all_strikes:
                moneyness = K / S0

                # Skip far-OTM / far-ITM strikes
                if not (0.50 <= moneyness <= 1.80):
                    continue

                # --- call IV ---
                c_row = calls[calls['strike'] == K]
                if not c_row.empty:
                    call_mid = float(c_row['mid'].iloc[0])
                    call_iv  = self._implied_vol(call_mid, S0, K, T_years, r, 'call')
                    call_price = self._bs_call(S0, K, T_years, r, call_iv) if not np.isnan(call_iv) else call_mid
                else:
                    call_iv    = np.nan
                    call_price = np.nan

                # --- put IV ---
                p_row = puts[puts['strike'] == K]
                if not p_row.empty:
                    put_mid  = float(p_row['mid'].iloc[0])
                    put_iv   = self._implied_vol(put_mid, S0, K, T_years, r, 'put')
                    put_price = self._bs_put(S0, K, T_years, r, put_iv) if not np.isnan(put_iv) else put_mid
                else:
                    put_iv    = np.nan
                    put_price = np.nan

                # Use call IV for OTM calls / ATM, put IV for OTM puts (market convention)
                if moneyness >= 1.0:
                    iv = call_iv if not np.isnan(call_iv) else put_iv
                else:
                    iv = put_iv if not np.isnan(put_iv) else call_iv

                if np.isnan(iv) or iv <= 0:
                    continue

                # Recompute consistent prices from IV
                call_price = self._bs_call(S0, K, T_years, r, iv)
                put_price  = self._bs_put(S0, K, T_years, r, iv)
                delta_call, gamma, vega, theta_call = self._greeks(S0, K, T_years, r, iv)

                rows.append({
                    'ticker':          ticker,
                    'valuation_date':  self.valuation_date,
                    'maturity_years':  round(T_years, 6),
                    'maturity_days':   T_days,
                    'strike':          K,
                    'moneyness':       round(moneyness, 6),
                    'spot_price':      S0,
                    'call_price':      call_price,
                    'put_price':       put_price,
                    'implied_vol':     iv,
                    'delta_call':      delta_call,
                    'gamma':           gamma,
                    'vega':            vega,
                    'theta_call':      theta_call,
                    'risk_free_rate':  r,
                })

        df = pd.DataFrame(rows)
        print(f"    Rows generated : {len(df)}")
        return df

    # ------------------------------------------------------------------
    # Common-expiry intersection
    # ------------------------------------------------------------------

    def _find_common_expiries(self) -> list:
        """
        Return the list of expiry strings available for ALL tickers,
        sorted by date.
        """
        print("\n  Fetching available expiries per ticker ...")
        sets = []
        for t in self.tickers:
            try:
                _, exps = self._fetch_expiries(t)
                s = set(exps)
                print(f"    {t}: {len(s)} expiries")
                sets.append(s)
            except Exception as e:
                print(f"    WARNING: could not fetch expiries for {t}: {e}")
                sets.append(set())

        if not sets:
            return []

        common = sets[0].intersection(*sets[1:])
        # Keep only future expiries
        today_str = date.today().isoformat()
        common = sorted(exp for exp in common if exp > today_str)
        print(f"  Common expiries ({len(common)}): {common[:5]}{'...' if len(common) > 5 else ''}")
        return common

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def generate_all_surfaces(self):
        """
        Main entry point.  Fetches live data and writes outputs identical
        in format to all_bs_surfaces.csv.
        """
        print("\n" + "=" * 60)
        print("BUILDING LIVE BLACK-SCHOLES SURFACES")
        print(f"Tickers       : {self.tickers}")
        print(f"Valuation date: {self.valuation_date}")
        print("=" * 60)

        common_expiries = self._find_common_expiries()
        if not common_expiries:
            raise RuntimeError("No common expiries found across tickers.")

        all_dfs = []
        for ticker in self.tickers:
            df = self._build_surface(ticker, common_expiries)
            if not df.empty:
                # Save per-ticker CSV
                path = os.path.join(self.output_dir, f"{ticker}_bs_surface.csv")
                df.to_csv(path, index=False)
                print(f"    Saved: {path}")
            all_dfs.append(df)

        combined = pd.concat(all_dfs, ignore_index=True)

        # Enforce same column order as the synthetic CSV
        col_order = [
            'ticker', 'valuation_date', 'maturity_years', 'maturity_days',
            'strike', 'moneyness', 'spot_price', 'call_price', 'put_price',
            'implied_vol', 'delta_call', 'gamma', 'vega', 'theta_call',
            'risk_free_rate'
        ]
        combined = combined[col_order]

        out_path = os.path.join(self.output_dir, 'all_bs_surfaces.csv')
        combined.to_csv(out_path, index=False)

        # Also build the calibration pickle
        calib = self._prepare_calibration_data(combined)
        pkl_path = os.path.join(self.output_dir, 'calibration_data.pkl')
        with open(pkl_path, 'wb') as f:
            pickle.dump(calib, f)

        print("\n" + "=" * 60)
        print("DONE")
        print(f"  Total rows          : {len(combined)}")
        print(f"  Common maturities   : {len(common_expiries)}")
        print(f"  Output directory    : {self.output_dir}/")
        print("    [TICKER]_bs_surface.csv  (per-ticker surfaces)")
        print("    all_bs_surfaces.csv      (combined — same format as synthetic)")
        print("    calibration_data.pkl     (for modelling)")
        print("=" * 60)

        return combined

    # ------------------------------------------------------------------
    # Calibration data preparation (mirrors synthetic generator)
    # ------------------------------------------------------------------

    def _calculate_vega(self, S, K_arr, T, r, sigma_arr):
        d1 = (np.log(S / K_arr) + (r + 0.5 * sigma_arr ** 2) * T) / (sigma_arr * np.sqrt(T))
        return S * norm.pdf(d1) * np.sqrt(T)

    def _prepare_calibration_data(self, combined: pd.DataFrame) -> dict:
        print("\n  Building calibration data ...")
        calib = {
            'metadata': {
                'valuation_date':  self.valuation_date,
                'risk_free_rate':  self.risk_free_rate,
                'n_tickers':       len(self.tickers),
                'data_source':     'yfinance_live',
            },
            'heston':        {},
            'bs_smile':      {},
            'surface_stats': {},
            'validation':    {},
        }

        for ticker in self.tickers:
            td = combined[combined['ticker'] == ticker].copy()
            if td.empty:
                continue
            S0 = td['spot_price'].iloc[0]
            maturities = sorted(td['maturity_years'].unique())

            heston = {
                'spot':         S0,
                'maturities':   maturities,
                'strikes':      {},
                'prices':       {},
                'implied_vols': {},
                'weights':      {},
            }
            for T in maturities:
                sub = td[td['maturity_years'] == T].sort_values('strike')
                K_arr     = sub['strike'].values
                iv_arr    = sub['implied_vol'].values
                heston['strikes'][T]      = K_arr.tolist()
                heston['prices'][T]       = sub['call_price'].values.tolist()
                heston['implied_vols'][T] = iv_arr.tolist()
                vega = self._calculate_vega(S0, K_arr, T, self.risk_free_rate, iv_arr)
                vega_sum = vega.sum()
                heston['weights'][T] = (vega / vega_sum).tolist() if vega_sum > 0 else (np.ones_like(vega) / len(vega)).tolist()

            calib['heston'][ticker] = heston

            # Surface stats
            atm = td[np.abs(td['moneyness'] - 1.0) < 0.02]
            calib['surface_stats'][ticker] = {
                'min_vol':  td['implied_vol'].min(),
                'max_vol':  td['implied_vol'].max(),
                'atm_vols': atm.groupby('maturity_years')['implied_vol'].mean().to_dict(),
            }

            calib['validation'][ticker] = {
                'has_negative_prices': (td['call_price'] < 0).any(),
                'has_nan_vols':        td['implied_vol'].isna().any(),
                'n_options':           len(td),
                'n_maturities':        len(maturities),
            }

        return calib

    # ------------------------------------------------------------------
    # Validation (same interface as synthetic generator)
    # ------------------------------------------------------------------

    def validate_surface(self, df: pd.DataFrame) -> dict:
        print("\n" + "=" * 60)
        print("VALIDATING OPTION SURFACE")
        print("=" * 60)
        results = {}
        for ticker in self.tickers:
            td = df[df['ticker'] == ticker]
            for T in td['maturity_years'].unique():
                sub = td[td['maturity_years'] == T].sort_values('strike')
                prices = sub['call_price'].values
                is_monotonic = bool(np.all(np.diff(prices) <= 1e-4))
                is_convex    = bool(np.all(np.diff(prices, 2) >= -1e-4)) if len(prices) >= 3 else True
                results[f"{ticker}_T{round(T,4)}"] = {
                    'monotonic': is_monotonic,
                    'convex':    is_convex,
                }
        for k, v in results.items():
            print(f"  {k}: {v}")
        return results


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def main():
    generator = BlackScholesLiveSurface(
        tickers=['NFLX', 'SPOT', 'DIS'],
        risk_free_rate=0.05,
        output_dir='../../final_dataset/live_option/black_scholes',
        min_volume=0,           # increase to filter illiquid contracts
        min_open_interest=0,    # e.g. set to 10 to require some open interest
    )

    option_data = generator.generate_all_surfaces()
    generator.validate_surface(option_data)

    print("\n" + "=" * 60)
    print("SAMPLE — near-ATM options")
    print("=" * 60)
    sample = option_data[
        (option_data['moneyness'] >= 0.95) & (option_data['moneyness'] <= 1.05)
    ].head(15)
    print(sample[['ticker', 'maturity_years', 'maturity_days', 'strike',
                  'moneyness', 'call_price', 'put_price', 'implied_vol',
                  'delta_call']].to_string(index=False))


if __name__ == "__main__":
    main()


BUILDING LIVE BLACK-SCHOLES SURFACES
Tickers       : ['NFLX', 'SPOT', 'DIS']
Valuation date: 2026-03-11

  Fetching available expiries per ticker ...
    NFLX: 21 expiries
    SPOT: 19 expiries
    DIS: 20 expiries
  Common expiries (18): ['2026-03-13', '2026-03-20', '2026-03-27', '2026-04-02', '2026-04-10']...

  Processing NFLX ...
    Spot price : 96.94
    Rows generated : 1439
    Saved: ../../final_dataset/live_option/black_scholes\NFLX_bs_surface.csv

  Processing SPOT ...
    Spot price : 530.26
    Rows generated : 1043
    Saved: ../../final_dataset/live_option/black_scholes\SPOT_bs_surface.csv

  Processing DIS ...
    Spot price : 101.32
    Rows generated : 517
    Saved: ../../final_dataset/live_option/black_scholes\DIS_bs_surface.csv

  Building calibration data ...

DONE
  Total rows          : 2999
  Common maturities   : 18
  Output directory    : ../../final_dataset/live_option/black_scholes/
    [TICKER]_bs_surface.csv  (per-ticker surfaces)
    all_bs_surfaces.csv

In [35]:
import yfinance as yf

ticker = yf.Ticker("DIS")

expirations = ticker.options
print(expirations)

opt_chain = ticker.option_chain(expirations[0])

calls = opt_chain.calls
puts = opt_chain.puts

print(calls.head())

('2026-03-13', '2026-03-20', '2026-03-27', '2026-04-02', '2026-04-10', '2026-04-17', '2026-04-24', '2026-05-15', '2026-06-18', '2026-07-17', '2026-08-21', '2026-09-18', '2026-10-16', '2026-11-20', '2026-12-18', '2027-01-15', '2027-03-19', '2027-06-17', '2027-12-17', '2028-01-21')
       contractSymbol             lastTradeDate  strike  lastPrice  bid  ask  \
0  DIS260313C00065000 2026-03-09 14:35:31+00:00    65.0      34.61  0.0  0.0   
1  DIS260313C00070000 2026-03-10 17:53:30+00:00    70.0      32.00  0.0  0.0   
2  DIS260313C00075000 2026-03-09 14:36:05+00:00    75.0      24.47  0.0  0.0   
3  DIS260313C00080000 2026-03-09 14:36:05+00:00    80.0      19.50  0.0  0.0   
4  DIS260313C00085000 2026-03-10 19:35:28+00:00    85.0      16.57  0.0  0.0   

   change  percentChange  volume  openInterest  impliedVolatility  inTheMoney  \
0     0.0            0.0     2.0             0            0.00001        True   
1     0.0            0.0   111.0             0            0.00001        Tru

In [42]:
"""
Heston Model Live Option Surface Builder
Fetches real options data from yfinance and computes implied volatility surfaces
with Heston-style parameterization.
Output format matches all_heston_surfaces.csv produced by the synthetic generator.
"""

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize
from scipy.stats import norm
from datetime import datetime, date
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

try:
    import yfinance as yf
except ImportError:
    raise ImportError("yfinance is required. Install with: pip install yfinance")


class HestonLiveSurface:
    """
    Build Heston-style implied volatility surfaces from real options data via yfinance.
    Output matches the format of all_heston_surfaces.csv produced by the synthetic generator.
    """

    def __init__(self,
                 tickers: list = None,
                 risk_free_rate: float = 0.05,
                 output_dir: str = '../../final_dataset/live_option/heston',
                 min_volume: int = 0,
                 min_open_interest: int = 0):
        """
        Parameters
        ----------
        tickers           : list of Yahoo Finance ticker symbols
        risk_free_rate    : annualised risk-free rate (e.g. 0.05 = 5 %)
        output_dir        : directory for CSV / pickle outputs
        min_volume        : minimum daily volume filter on option contracts
        min_open_interest : minimum open interest filter on option contracts
        """
        self.tickers = tickers or ['NFLX', 'SPOT', 'DIS']
        self.risk_free_rate = risk_free_rate
        self.output_dir = output_dir
        self.min_volume = min_volume
        self.min_open_interest = min_open_interest
        self.valuation_date = date.today().isoformat()

        os.makedirs(output_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # Black-Scholes helpers (used for IV extraction from market prices)
    # ------------------------------------------------------------------

    def _bs_call(self, S, K, T, r, sigma):
        if T <= 0 or sigma <= 0:
            return max(0.0, S - K)
        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def _bs_put(self, S, K, T, r, sigma):
        return self._bs_call(S, K, T, r, sigma) - S + K * np.exp(-r * T)

    def _implied_vol(self, market_price, S, K, T, r, option_type='call'):
        """Brent-method implied volatility from market mid-price."""
        if T <= 0 or market_price <= 0:
            return np.nan
        intrinsic = max(0.0, S - K) if option_type == 'call' else max(0.0, K - S)
        if market_price < intrinsic - 1e-4:
            return np.nan

        def obj(sigma):
            if option_type == 'call':
                return self._bs_call(S, K, T, r, sigma) - market_price
            return self._bs_put(S, K, T, r, sigma) - market_price

        try:
            return brentq(obj, 1e-4, 5.0, xtol=1e-6, maxiter=200)
        except Exception:
            return np.nan

    # ------------------------------------------------------------------
    # Heston-style vol surface parameterization
    # ------------------------------------------------------------------

    def heston_vol_surface(self, T, m, params):
        """
        Approximate Heston implied vol surface.
        Mirrors the synthetic generator's formula exactly so downstream
        calibration code sees the same functional form.
        """
        v0    = params['v0']
        theta = params['theta']
        kappa = params['kappa']
        rho   = params['rho']
        sigma = params['sigma']

        if T <= 0:
            return np.sqrt(v0)

        decay = (1 - np.exp(-kappa * T)) / (kappa * T)

        vol_T      = np.sqrt(theta + (v0 - theta) * np.exp(-kappa * T))
        skew_effect = rho * sigma * decay * np.log(m)
        convexity   = 0.1 * sigma * decay * (np.log(m)) ** 2

        iv = vol_T + skew_effect + convexity
        return float(np.clip(iv, 0.05, 2.0))

    # ------------------------------------------------------------------
    # Heston parameter fitting from observed IV surface
    # ------------------------------------------------------------------

    def _fit_heston_params(self, surface_rows: list, S0: float) -> dict:
        """
        Fit Heston parameters (v0, kappa, theta, sigma, rho) to the
        observed implied vol surface via least-squares minimisation.

        Falls back to sensible defaults if optimisation fails.
        """
        default = {'v0': 0.04, 'kappa': 2.0, 'theta': 0.05, 'sigma': 0.3, 'rho': -0.6}
        if not surface_rows:
            return default

        Ts  = np.array([r['T']  for r in surface_rows])
        ms  = np.array([r['m']  for r in surface_rows])
        ivs = np.array([r['iv'] for r in surface_rows])

        def residuals(x):
            v0, kappa, theta, sigma, rho = x
            if any([v0 <= 0, kappa <= 0, theta <= 0, sigma <= 0,
                    not (-1 < rho < 1)]):
                return 1e6
            params = dict(v0=v0, kappa=kappa, theta=theta, sigma=sigma, rho=rho)
            pred = np.array([self.heston_vol_surface(T, m, params)
                             for T, m in zip(Ts, ms)])
            return float(np.mean((pred - ivs) ** 2))

        x0 = [default['v0'], default['kappa'], default['theta'],
              default['sigma'], default['rho']]
        bounds = [(1e-4, 1.0), (0.1, 10.0), (1e-4, 1.0),
                  (1e-4, 2.0), (-0.99, 0.99)]

        try:
            res = minimize(residuals, x0, method='L-BFGS-B', bounds=bounds,
                           options={'maxiter': 500, 'ftol': 1e-10})
            v0, kappa, theta, sigma, rho = res.x
            return dict(v0=v0, kappa=kappa, theta=theta, sigma=sigma, rho=rho)
        except Exception:
            return default

    # ------------------------------------------------------------------
    # yfinance data fetch helpers
    # ------------------------------------------------------------------

    def _fetch_spot(self, ticker: str) -> float:
        tk = yf.Ticker(ticker)
        hist = tk.history(period='1d')
        if hist.empty:
            raise ValueError(f"Could not fetch spot price for {ticker}")
        return float(hist['Close'].iloc[-1])

    def _fetch_expiries(self, ticker: str):
        tk = yf.Ticker(ticker)
        return tk, list(tk.options)

    def _expiry_to_years(self, expiry_str: str):
        exp   = datetime.strptime(expiry_str, '%Y-%m-%d').date()
        today = date.today()
        days  = (exp - today).days
        return days / 365.0, days

    # ------------------------------------------------------------------
    # Common-expiry intersection (same logic as blackScholes_live.py)
    # ------------------------------------------------------------------

    def _find_common_expiries(self) -> list:
        print("\n  Fetching available expiries per ticker ...")
        sets = []
        for t in self.tickers:
            try:
                _, exps = self._fetch_expiries(t)
                s = set(exps)
                print(f"    {t}: {len(s)} expiries")
                sets.append(s)
            except Exception as e:
                print(f"    WARNING: could not fetch expiries for {t}: {e}")
                sets.append(set())

        if not sets:
            return []

        common = sets[0].intersection(*sets[1:])
        today_str = date.today().isoformat()
        common = sorted(exp for exp in common if exp > today_str)
        print(f"  Common expiries ({len(common)}): "
              f"{common[:5]}{'...' if len(common) > 5 else ''}")
        return common

    # ------------------------------------------------------------------
    # Core surface builder for one ticker
    # ------------------------------------------------------------------

    def generate_option_surface(self, ticker: str, common_expiries: list):
        """
        Fetch option chains for `common_expiries`, extract market IVs,
        fit Heston parameters, then populate the output rows.
        Returns (DataFrame, heston_params_dict).
        """
        print(f"\n  Processing {ticker} ...")
        tk = yf.Ticker(ticker)
        S0 = self._fetch_spot(ticker)
        r  = self.risk_free_rate
        print(f"    Spot price : {S0:.2f}")

        # ---- pass 1: collect raw IV points for Heston fitting --------
        raw_iv_points = []
        chain_cache   = {}   # expiry -> (calls_df, puts_df)

        for expiry in common_expiries:
            T_years, T_days = self._expiry_to_years(expiry)
            if T_years <= 0:
                continue

            try:
                chain = tk.option_chain(expiry)
            except Exception as e:
                print(f"    WARNING: could not fetch chain for {expiry}: {e}")
                continue

            calls = chain.calls.copy()
            puts  = chain.puts.copy()

            if self.min_volume > 0:
                calls = calls[calls['volume'].fillna(0) >= self.min_volume]
                puts  = puts[puts['volume'].fillna(0) >= self.min_volume]
            if self.min_open_interest > 0:
                calls = calls[calls['openInterest'].fillna(0) >= self.min_open_interest]
                puts  = puts[puts['openInterest'].fillna(0) >= self.min_open_interest]

            def mid(df):
                bid = df['bid'].fillna(0)
                ask = df['ask'].fillna(0)
                return np.where((bid > 0) & (ask > 0),
                                (bid + ask) / 2, df['lastPrice'])

            calls = calls.copy(); calls['mid'] = mid(calls)
            puts  = puts.copy();  puts['mid']  = mid(puts)
            chain_cache[expiry] = (calls, puts, T_years, T_days)

            # sample IV points for fitting
            all_strikes = sorted(set(calls['strike']) | set(puts['strike']))
            for K in all_strikes:
                m = K / S0
                if not (0.60 <= m <= 1.50):
                    continue
                if m >= 1.0:
                    row = calls[calls['strike'] == K]
                    otype = 'call'
                else:
                    row = puts[puts['strike'] == K]
                    otype = 'put'

                if row.empty:
                    continue
                price = float(row['mid'].iloc[0])
                iv = self._implied_vol(price, S0, K, T_years, r, otype)
                if not np.isnan(iv) and iv > 0:
                    raw_iv_points.append({'T': T_years, 'm': m, 'iv': iv})

        # ---- fit Heston parameters to observed IV surface -------------
        params = self._fit_heston_params(raw_iv_points, S0)
        print(f"    Heston fit → v0={params['v0']:.4f}  kappa={params['kappa']:.3f}  "
              f"theta={params['theta']:.4f}  sigma={params['sigma']:.3f}  "
              f"rho={params['rho']:.3f}")

        # ---- pass 2: build output rows using Heston-smoothed IVs -----
        rows = []
        for expiry, (calls, puts, T_years, T_days) in chain_cache.items():
            all_strikes = sorted(set(calls['strike']) | set(puts['strike']))

            for K in all_strikes:
                m = K / S0
                if not (0.50 <= m <= 1.80):
                    continue

                # Market IV (for reference / fallback)
                if m >= 1.0:
                    row = calls[calls['strike'] == K]
                    otype = 'call'
                else:
                    row = puts[puts['strike'] == K]
                    otype = 'put'

                market_iv = np.nan
                if not row.empty:
                    price = float(row['mid'].iloc[0])
                    market_iv = self._implied_vol(price, S0, K, T_years, r, otype)

                # Heston-smoothed IV (mirrors synthetic formula)
                heston_iv = self.heston_vol_surface(T_years, m, params)

                # Use market IV where available, Heston surface as fallback
                iv = market_iv if (not np.isnan(market_iv) and market_iv > 0) else heston_iv
                if iv <= 0:
                    continue

                call_price = self._bs_call(S0, K, T_years, r, iv)

                rows.append({
                    'ticker':          ticker,
                    'valuation_date':  self.valuation_date,
                    'maturity_years':  round(T_years, 6),
                    'maturity_days':   T_days,
                    'strike':          K,
                    'moneyness':       round(m, 6),
                    'spot_price':      S0,
                    'call_price':      call_price,
                    'implied_vol':     iv,
                    'heston_v0':       params['v0'],
                    'heston_kappa':    params['kappa'],
                    'heston_theta':    params['theta'],
                    'heston_sigma':    params['sigma'],
                    'heston_rho':      params['rho'],
                    'risk_free_rate':  r,
                })

        df = pd.DataFrame(rows)
        print(f"    Rows generated : {len(df)}")

        path = os.path.join(self.output_dir, f"{ticker}_heston_surface.csv")
        df.to_csv(path, index=False)
        print(f"    Saved: {path}")
        return df, params

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def generate_all_surfaces(self):
        """
        Main entry point. Fetches live data, fits Heston parameters per
        ticker, and writes outputs identical in format to all_heston_surfaces.csv.
        """
        print("\n" + "=" * 60)
        print("BUILDING LIVE HESTON SURFACES")
        print(f"Tickers       : {self.tickers}")
        print(f"Valuation date: {self.valuation_date}")
        print("=" * 60)

        common_expiries = self._find_common_expiries()
        if not common_expiries:
            raise RuntimeError("No common expiries found across tickers.")

        all_dfs   = []
        all_params = {}

        for ticker in self.tickers:
            df, params = self.generate_option_surface(ticker, common_expiries)
            all_dfs.append(df)
            all_params[ticker] = params

        combined = pd.concat(all_dfs, ignore_index=True)

        # Enforce same column order as the synthetic CSV
        col_order = [
            'ticker', 'valuation_date', 'maturity_years', 'maturity_days',
            'strike', 'moneyness', 'spot_price', 'call_price', 'implied_vol',
            'heston_v0', 'heston_kappa', 'heston_theta', 'heston_sigma',
            'heston_rho', 'risk_free_rate'
        ]
        combined = combined[col_order]

        out_path = os.path.join(self.output_dir, 'all_heston_surfaces.csv')
        combined.to_csv(out_path, index=False)

        with open(os.path.join(self.output_dir, 'heston_parameters.pkl'), 'wb') as f:
            pickle.dump(all_params, f)

        calib = self.prepare_calibration_data(combined)
        with open(os.path.join(self.output_dir, 'calibration_data.pkl'), 'wb') as f:
            pickle.dump(calib, f)

        print("\n" + "=" * 60)
        print("DONE")
        print(f"  Total rows          : {len(combined)}")
        print(f"  Common maturities   : {len(common_expiries)}")
        print(f"  Output directory    : {self.output_dir}/")
        print("    [TICKER]_heston_surface.csv  (per-ticker surfaces)")
        print("    all_heston_surfaces.csv      (combined — same format as synthetic)")
        print("    heston_parameters.pkl        (fitted Heston params per ticker)")
        print("    calibration_data.pkl         (for modelling)")
        print("=" * 60)

        return combined, all_params

    # ------------------------------------------------------------------
    # Calibration data preparation (mirrors synthetic generator)
    # ------------------------------------------------------------------

    def prepare_calibration_data(self, option_data: pd.DataFrame) -> dict:
        print("\n  Building calibration data ...")
        calib = {
            'metadata': {
                'valuation_date': self.valuation_date,
                'risk_free_rate': self.risk_free_rate,
                'n_tickers':      len(self.tickers),
                'data_source':    'yfinance_live',
            },
            'heston':        {},
            'surface_stats': {},
        }

        for ticker in self.tickers:
            td = option_data[option_data['ticker'] == ticker].copy()
            if td.empty:
                continue
            S0 = td['spot_price'].iloc[0]

            atm_vols = (td[np.abs(td['moneyness'] - 1.0) < 0.02]
                        .groupby('maturity_years')['implied_vol'].mean())

            skew_by_mat = {}
            conv_by_mat = {}
            for T in td['maturity_years'].unique():
                sub = td[td['maturity_years'] == T]
                v90  = sub[np.abs(sub['moneyness'] - 0.90) < 0.02]['implied_vol'].mean()
                v100 = sub[np.abs(sub['moneyness'] - 1.00) < 0.02]['implied_vol'].mean()
                v110 = sub[np.abs(sub['moneyness'] - 1.10) < 0.02]['implied_vol'].mean()
                if not any(np.isnan([v90, v100, v110])):
                    skew_by_mat[T] = v90 - v110
                    conv_by_mat[T] = v90 + v110 - 2 * v100

            calib['heston'][ticker] = {
                'spot':       S0,
                'maturities': sorted(td['maturity_years'].unique()),
                'params': {
                    'v0':    td['heston_v0'].iloc[0],
                    'kappa': td['heston_kappa'].iloc[0],
                    'theta': td['heston_theta'].iloc[0],
                    'sigma': td['heston_sigma'].iloc[0],
                    'rho':   td['heston_rho'].iloc[0],
                },
                'options': td.to_dict('records'),
            }

            calib['surface_stats'][ticker] = {
                'n_options':            len(td),
                'n_maturities':         td['maturity_years'].nunique(),
                'atm_vols':             atm_vols.to_dict(),
                'skew_by_maturity':     skew_by_mat,
                'convexity_by_maturity': conv_by_mat,
                'avg_skew':     float(np.mean(list(skew_by_mat.values()))) if skew_by_mat else 0.0,
                'avg_convexity': float(np.mean(list(conv_by_mat.values()))) if conv_by_mat else 0.0,
            }

            print(f"\n  {ticker} Calibration Data:")
            print(f"    Options     : {len(td)}")
            print(f"    Maturities  : {td['maturity_years'].nunique()}")
            if len(atm_vols) > 0:
                print(f"    ATM Vol     : {atm_vols.min():.3f} – {atm_vols.max():.3f}")
            print(f"    Avg Skew    : {calib['surface_stats'][ticker]['avg_skew']:.3f}")
            print(f"    Avg Convex  : {calib['surface_stats'][ticker]['avg_convexity']:.3f}")

        return calib

    # ------------------------------------------------------------------
    # Analysis / validation (same interface as synthetic generator)
    # ------------------------------------------------------------------

    def analyze_volatility_smile(self, df: pd.DataFrame):
        print("\n" + "=" * 60)
        print("VOLATILITY SMILE ANALYSIS")
        print("=" * 60)
        for ticker in self.tickers:
            td = df[df['ticker'] == ticker]
            print(f"\n{ticker} Volatility Smile:")
            for T in sorted(td['maturity_years'].unique())[:3]:
                sub = td[td['maturity_years'] == T].sort_values('moneyness')
                print(f"\n  Maturity {T:.4f}Y  ({int(T*365)}d):")
                for _, row in sub.iterrows():
                    print(f"    Moneyness {row['moneyness']:.3f}: IV={row['implied_vol']:.3f}")

    def verify_no_arbitrage(self, df: pd.DataFrame) -> bool:
        print("\n" + "=" * 60)
        print("VERIFYING NO-ARBITRAGE CONDITIONS")
        print("=" * 60)
        violations = []
        for ticker in self.tickers:
            td = df[df['ticker'] == ticker]
            for T in td['maturity_years'].unique():
                sub = td[td['maturity_years'] == T].sort_values('strike')
                prices = sub['call_price'].values
                if np.any(prices < -1e-6):
                    violations.append(f"{ticker} T={T:.4f}: Negative prices")
                if not np.all(np.diff(prices) <= 1e-4):
                    violations.append(f"{ticker} T={T:.4f}: Prices not monotonic")
                if len(prices) >= 3 and not np.all(np.diff(prices, 2) >= -0.05):
                    violations.append(f"{ticker} T={T:.4f}: Prices not convex")

        if not violations:
            print("  All no-arbitrage conditions satisfied.")
            return True
        print(f"  {len(violations)} violation(s) found:")
        for v in violations[:10]:
            print(f"    - {v}")
        if len(violations) > 10:
            print(f"    ... and {len(violations) - 10} more")
        return False


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def main():
    generator = HestonLiveSurface(
        tickers=['NFLX', 'SPOT', 'DIS'],
        risk_free_rate=0.05,
        output_dir='../../final_dataset/live_option/heston',
        min_volume=0,
        min_open_interest=0,
    )

    option_data, params = generator.generate_all_surfaces()
    generator.analyze_volatility_smile(option_data)
    generator.verify_no_arbitrage(option_data)

    print("\n" + "=" * 60)
    print("SAMPLE OPTION DATA (near-ATM Options)")
    print("=" * 60)
    sample = option_data[
        (option_data['moneyness'] >= 0.95) & (option_data['moneyness'] <= 1.05)
    ].head(15)
    print(sample[['ticker', 'maturity_years', 'maturity_days', 'strike',
                  'moneyness', 'call_price', 'implied_vol',
                  'heston_v0', 'heston_rho']].to_string(index=False))


if __name__ == "__main__":
    main()


BUILDING LIVE HESTON SURFACES
Tickers       : ['NFLX', 'SPOT', 'DIS']
Valuation date: 2026-03-11

  Fetching available expiries per ticker ...
    NFLX: 21 expiries
    SPOT: 19 expiries
    DIS: 20 expiries
  Common expiries (18): ['2026-03-13', '2026-03-20', '2026-03-27', '2026-04-02', '2026-04-10']...

  Processing NFLX ...
    Spot price : 96.94
    Heston fit → v0=0.5382  kappa=10.000  theta=0.1426  sigma=2.000  rho=-0.181
    Rows generated : 1441
    Saved: ../../final_dataset/live_option/heston\NFLX_heston_surface.csv

  Processing SPOT ...
    Spot price : 530.26
    Heston fit → v0=0.6858  kappa=10.000  theta=0.1982  sigma=2.000  rho=-0.480
    Rows generated : 1048
    Saved: ../../final_dataset/live_option/heston\SPOT_heston_surface.csv

  Processing DIS ...
    Spot price : 101.32
    Heston fit → v0=0.3712  kappa=10.000  theta=0.0853  sigma=0.000  rho=-0.347
    Rows generated : 521
    Saved: ../../final_dataset/live_option/heston\DIS_heston_surface.csv

  Building cali

In [31]:
from datetime import datetime

date_string = "2026-3-10"
format_string = "%Y-%m-%d"
date_object = datetime.strptime(date_string, format_string)

In [32]:
date_object2 = datetime.strptime(expirations[0], format_string) 

In [33]:
date_object2

datetime.datetime(2026, 3, 13, 0, 0)

In [28]:
(date_object2 - date_object).days

3

In [34]:
for e in expirations:
    date_object2 = datetime.strptime(e, format_string) 
    print((date_object2 - date_object).days)

3
10
17
23
31
38
45
66
100
129
164
192
255
283
311
374
464
647
682
829
1011


In [29]:
for e in expirations:
    date_object2 = datetime.strptime(e, format_string) 
    print((date_object2 - date_object).days)

3
10
17
23
31
38
45
66
100
129
164
192
220
255
283
311
374
464
647
682


In [36]:
maturities = []
for e in expirations:
    date_object2 = datetime.strptime(e, format_string) 
    maturities.append((date_object2 - date_object).days)

In [38]:
print(maturities)

[3, 10, 17, 23, 31, 38, 45, 66, 100, 129, 164, 192, 220, 255, 283, 311, 374, 464, 647, 682]
